# Renderizar os vídeos da fila

Este caderno existe para quando o GitHub Actions não entrega máquina: os
vídeos ficam parados em *na fila* e não há erro nenhum para consertar. Aqui
o Google empresta um computador de graça, com ffmpeg, e a renderização sai
por ele.

**Como usar, pelo iPhone:**

1. Toque no ▶ da célula abaixo.
2. Cole `DATABASE_URL` e `AUTH_SECRET` quando pedir — os mesmos valores que
   estão no painel da Vercel. O que você digita aqui não fica salvo no
   arquivo.
3. Espere. A primeira vez leva uns 5 minutos (instalar Node e as
   dependências); as seguintes, menos.
4. Quando terminar, abra a aba **Fila** no painel: os vídeos estarão prontos
   para aprovar e publicar.

Para não digitar as chaves toda vez, cadastre as duas no cofre do Colab
(ícone de chave 🔑 na barra da esquerda, com *Notebook access* ligado). O
caderno lê de lá sozinho.

> Mantenha a aba aberta enquanto roda. O Colab desliga a máquina quando você
> fecha, e uma renderização interrompida volta para a fila.

In [ ]:
# @title Renderizar os vídeos da fila { display-mode: "form" }

import getpass, os, subprocess

REPO = "https://github.com/guisegredo18-bit/autotok-pessoal"
BRANCH = "claude/tiktok-trends-analysis-app-pihsab"
PASTA = "/content/autotok"


def segredo(nome):
    """Cofre do Colab primeiro; se não estiver lá, pergunta.

    `getpass` em vez de `input` de propósito: o valor não aparece na tela nem
    fica gravado na saída da célula, que é salva junto com o caderno.
    """
    try:
        from google.colab import userdata

        valor = userdata.get(nome)
        if valor:
            return valor
    except Exception:
        pass
    return getpass.getpass(nome + ": ")


os.environ["DATABASE_URL"] = segredo("DATABASE_URL")
os.environ["AUTH_SECRET"] = segredo("AUTH_SECRET")

# ffmpeg renderiza; a fonte desenha a legenda queimada no vídeo.
!apt-get -qq install -y ffmpeg fonts-dejavu-core

# O Colab às vezes já traz o Node, às vezes traz uma versão velha demais.
_v = subprocess.run("node --version", shell=True, capture_output=True, text=True)
_maior = int(_v.stdout.strip().lstrip("v").split(".")[0]) if _v.returncode == 0 else 0
if _maior < 20:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
    !apt-get install -qq -y nodejs

# Rodar o caderno duas vezes na mesma sessão não deve clonar de novo.
if os.path.isdir(PASTA):
    !git -C $PASTA fetch --depth 1 origin $BRANCH && git -C $PASTA reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch $BRANCH $REPO $PASTA

%cd $PASTA
!npm install --no-audit --no-fund

# `queue` pega todos os vídeos que estão esperando, um a um, e para sozinho
# quando a fila esvazia.
!npm run queue

print("\nPronto. Abra a aba Fila no painel para aprovar e publicar.")